# Supplementary Figure — clustering geometry of across-condition spatial archetypes

This notebook summarizes the older clustering analysis without rerunning the
expensive spectral-clustering scan.

It loads the cached clustering summary and saved per-archetype decoding table,
then visualizes whether high-clustering archetypes also tend to carry stronger
decoding information.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CLUSTER_CSV = Path(
    "003_postprocess_across_spatial_cache/cluster_summary_df.csv"
)

DECODING_CSV = Path(
    "msaa_condrank_decoding_outputs_spatial_across/"
    "per_archetype_mean_accuracy.csv"
)

OUTPUT_DIR = Path("outputs/paper_supplement")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_PNG = OUTPUT_DIR / "Supplement_clustering_geometry.png"
FIG_PDF = OUTPUT_DIR / "Supplement_clustering_geometry.pdf"

for p in [CLUSTER_CSV, DECODING_CSV]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing {p}. Run 003_postprocess_across_spatial and "
            "the per-archetype decoding analysis first."
        )

cluster = pd.read_csv(CLUSTER_CSV)
decode = pd.read_csv(DECODING_CSV)

if "mean_accuracy" in decode.columns and "mean" not in decode.columns:
    decode = decode.rename(columns={"mean_accuracy": "mean"})

display(cluster.head())
display(decode.head())

In [ ]:
# Mean decoding over conditions for each K/archetype.
decode_mean = (
    decode
    .groupby(["K", "archetype"], as_index=False)
    .agg(decoding_mean=("mean", "mean"))
)

merged = cluster.merge(
    decode_mean,
    on=["K", "archetype"],
    how="inner",
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(14.0, 4.5),
    gridspec_kw={"wspace": 0.30},
)

axA, axB, axC = axes

# A — decoding vs cluster score
for K, sub in merged.groupby("K"):
    axA.scatter(
        sub["decoding_mean"],
        sub["cluster_score"],
        s=28,
        alpha=0.65,
        label=f"K={int(K)}",
    )

axA.set_xlabel("Mean per-archetype decoding accuracy")
axA.set_ylabel("Clustering score")
axA.set_title("Decoding vs clustering", fontweight="bold")
axA.legend(frameon=False, fontsize=9)

# B — purity vs balance
for K, sub in merged.groupby("K"):
    axB.scatter(
        sub["purity"],
        sub["balance"],
        s=28,
        alpha=0.65,
        label=f"K={int(K)}",
    )

axB.set_xlabel("Cluster purity")
axB.set_ylabel("Cluster balance")
axB.set_title("Clustering geometry", fontweight="bold")

# C — best clustering score at each K
best = (
    cluster
    .groupby("K", as_index=False)
    .agg(
        best_cluster_score=("cluster_score", "max"),
        median_cluster_score=("cluster_score", "median"),
    )
    .sort_values("K")
)

axC.plot(
    best["K"],
    best["best_cluster_score"],
    marker="o",
    linewidth=1.8,
    label="Best archetype",
)
axC.plot(
    best["K"],
    best["median_cluster_score"],
    marker="s",
    linestyle="--",
    linewidth=1.5,
    label="Median archetype",
)
axC.set_xlabel("Model order K")
axC.set_ylabel("Clustering score")
axC.set_title("Clustering structure across model order", fontweight="bold")
axC.legend(frameon=False, fontsize=9)

for ax, letter in zip(axes, list("ABC")):
    ax.text(
        -0.12, 1.04, letter,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "Condition-related clustering structure in across-condition spatial MS-AA",
    fontsize=15,
    fontweight="bold",
)

fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)